In [ ]:
import os

import h5py
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots

folder = r"C:\DATA\StonyBrookCollab\2025_10_01_tpx\C10F18\C10F18_pumpprobe\finescan1_2025-10-01_13-55"
data = pd.DataFrame()
for file in os.listdir(folder):
    if not file.endswith('.cv4'):
        continue
    ind = file[:-4].split('_')[-1]
    print(ind)
    with h5py.File(os.path.join(folder, file), 'r') as f:
        x = f['x'][:]
        y = f['y'][:]
        t = f['t'][:]
    temp = pd.DataFrame({'x': x, 'y': y, 't': t, 'index': ind})
    data = pd.concat([data, temp], ignore_index=True)
data['index'] = data['index'].astype(int)

electron_data = data[(data['t'] <= 4000) & (data['t'] >= 3000)]
data = data[(data['t'] >= 4500) & (data['t'] < 10000)]
data = data.reset_index()
#sort by index
data = data.sort_values(by='index')

In [ ]:
hist, xe, ye = np.histogram2d(data['index'], data['t'], bins=[54, 1000], range=[[0, 53], [4100, 7600]])
#normalize each column
# for i in range(hist.shape[0]):
#     hist[i,:]=hist[i,:]/np.sum(hist[i,:])

px.histogram(data, x='t', nbins=1000, title='ToF Spectrum', width=600, height=400, log_y=True).write_html(
        "tof_spectrum.html")

px.imshow(np.log(hist.T), x=xe[:-1], y=ye[:-1], aspect='auto', origin='lower',
          labels={'y': 'etof (a.u.)', 'x': 'file index'}, title='tof vs file index').write_html("time_vs_index.html")

In [ ]:
calibration_points = [(7.71, 80), (5.77, 15), (6.68, 39)]
x_cal, y_cal = zip(*calibration_points)

fit = np.polyfit(x_cal, y_cal, 2)
p = np.poly1d(fit)

data['m/q'] = p(data['t'] / 1000 + 0.4)

px.histogram(data.sample(frac=0.1), x='m/q', nbins=1000, title='Mass Spectrum', width=600, height=400,
             log_y=True).write_html(
        "mass_spectrum.html"
)

In [ ]:
hist2, xe2, ye2 = np.histogram2d(data['index'], data['m/q'], bins=[54, 100], range=[[0, 53], [0, 65]])
px.imshow(np.log(hist2.T), x=xe2[:-1], y=ye2[:-1], aspect='auto', origin='lower',
          labels={'y': 'm/q (a.u.)', 'x': 'file index'}, title='m/q vs file index').write_html("mass_vs_index.html")

total_counts = hist2.sum(axis=1)

hist1d = np.histogram(data['m/q'], bins=100, range=(0, 65))[0]
hist1d = hist1d / np.sum(hist1d)

expected_counts = np.outer(total_counts, hist1d)

diff = (hist2 - expected_counts)  # +1 to avoid division by zero

px.imshow(diff.T, x=xe2[:-1], y=ye2[:-1], aspect='auto', origin='lower',
          labels={'y': 'm/q (a.u.)', 'x': 'file index'}, title='Difference between observed and expected counts',
          color_continuous_scale=px.colors.sequential.RdBu_r, color_continuous_midpoint=0
          ).write_html("diff_mass_vs_index.html")

#show row normalized difference
row_sums = hist2.sum(axis=1, keepdims=True)
row_normalized_diff = diff / row_sums


def apply_gamma_correction(arr, gamma=0.5):
    sign = np.sign(arr)
    magnitude = np.abs(arr) ** gamma
    return sign * magnitude


px.imshow(apply_gamma_correction(row_normalized_diff.T, gamma=0.5), x=xe2[:-1], y=ye2[:-1], aspect='auto',
          origin='lower',
          labels={'y': 'm/q (a.u.)', 'x': 'file index'},
          title='Row Normalized Difference between observed and expected counts',
          color_continuous_scale=px.colors.sequential.RdBu_r, color_continuous_midpoint=0,
          ).write_html("row_normalized_diff_mass_vs_index.html")


In [ ]:
#plot counts vs index

counts = data.groupby('index').size()
px.line(counts, title='Counts vs File Index', labels={'index': 'File Index', 0: 'Counts'}).write_html(
        "counts_vs_index.html")

In [ ]:
t0 = 28
time_step = 10

data['time_delay'] = (data['index'] - t0) * time_step



In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Enhancement', 'Counts vs Time Delay'), )
fig.add_trace(
        px.imshow(
                apply_gamma_correction(diff.T, gamma=0.5),
                x=(xe2[:-1] - t0) * time_step,
                y=ye2[:-1],
                aspect="auto",
                origin="lower",
                labels={"y": "m/q (a.u.)", "x": "File Index", "color": "Row Normalized Diff"},
                color_continuous_scale=px.colors.sequential.RdBu_r,
                color_continuous_midpoint=0,
        ).data[0],
        row=1, col=1
)
fig.add_trace(
        px.line(x=counts.index * time_step - t0 * time_step, y=counts.values,
                labels={'x': 'Delay (fs)', 'y': 'Counts'}).data[0],
        row=2, col=1
)
fig.update_layout(height=800, width=600,
                  coloraxis_colorscale=px.colors.sequential.RdBu_r, coloraxis_cmid=0,
                  coloraxis_colorbar=dict(title='Enhancement'),
                  )
fig.update_xaxes(title_text='Delay (fs)', row=2, col=1)
fig.update_yaxes(title_text='m/q (a.u.)', row=1, col=1)
fig.update_yaxes(title_text='Counts', row=2, col=1)
fig.write_html("mass_and_counts_vs_delay.html")



In [ ]:
electron_data['time_delay'] = (electron_data['index'] - t0) * time_step
counts_electron = electron_data.groupby('time_delay').size()
px.line(counts_electron, title='Electron Counts vs Time Delay', labels={'index': 'Delay (fs)', 0: 'Counts'}).write_html(
        "electron_counts_vs_delay.html")